# W&B 项目复制工具：eval_log → eval_all

将 `eval_log` 中所有 run 的 `summary/*` 表格数据（history 中以 `summary/` 开头的列）
复制到 `eval_all` 项目，保持 run 名、epoch 对齐不变。

**排除**：`summary/IJBC_TPR@FPR0.01`

需要在 `conda` 的 `zkj-work` 环境里运行。

In [1]:
import math
import sys
from typing import List, Set

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [2]:
# ── 配置 ──────────────────────────────────────────────────────────
ENTITY = 'kejian-zhao-tsinghua-university'
SRC_PROJECT = 'eval_log'
DST_PROJECT = 'eval_all'

# 只复制以此开头的 history 列 (即 W&B 面板上的表格/折线图)
COPY_PREFIX = 'summary/'

# 要排除的列名 (精确匹配)
EXCLUDE_COLUMNS = {
    # 'summary/IJBC_TPR@FPR0.01',
}

# 横轴: epoch
STEP_METRIC = 'epoch'

print(f'源项目  : {ENTITY}/{SRC_PROJECT}')
print(f'目标项目: {ENTITY}/{DST_PROJECT}')
print(f'复制前缀: {COPY_PREFIX}*')
print(f'排除列  : {EXCLUDE_COLUMNS}')
print(f'横轴    : {STEP_METRIC}')

源项目  : kejian-zhao-tsinghua-university/eval_log
目标项目: kejian-zhao-tsinghua-university/eval_all
复制前缀: summary/*
排除列  : {}
横轴    : epoch


In [3]:
# ── 核心工具函数 ────────────────────────────────────────────────────

def is_missing(value):
    """判断值是否为缺失值 (None / NaN)。"""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    try:
        m = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(m) if isinstance(m, bool) else False


def get_summary_columns(run):
    """
    获取 run 中所有 summary/* 开头的列名 (排除 EXCLUDE_COLUMNS)。
    用 run.history(samples=1) 快速拿到列名列表。
    """
    df = run.history(samples=1)
    return sorted([
        c for c in df.columns
        if c.startswith(COPY_PREFIX) and c not in EXCLUDE_COLUMNS
    ])


def fetch_summary_table(run):
    """
    从 run 的 history 中提取 summary/* 列 + epoch 列。
    使用 scan_history(keys=...) 服务端过滤, 避免拉取无关行。

    返回:
        df        : 过滤后的 DataFrame (每行 = 一个 epoch)
        data_cols : summary/* 数据列名列表
    """
    data_cols = get_summary_columns(run)
    if not data_cols:
        return pd.DataFrame(), []

    rows = list(run.scan_history(keys=[STEP_METRIC] + data_cols))
    if not rows:
        return pd.DataFrame(), []

    df = pd.DataFrame(rows)
    # 只保留需要的列 (scan_history 可能返回额外字段)
    keep = [c for c in [STEP_METRIC] + data_cols if c in df.columns]
    df = df[keep]
    df = df.dropna(subset=data_cols, how='all').reset_index(drop=True)

    return df, data_cols


def preview_run(src_run, dst_exists):
    """预览一个 run 的复制计划, 不执行写入。"""
    df, data_cols = fetch_summary_table(src_run)

    print('=' * 72)
    print(f'  run: {src_run.name} ({src_run.id})')
    print(f'  dst: {"resume" if dst_exists else "create new"}')
    print(f'  epoch 数: {len(df)}')
    print(f'  data columns ({len(data_cols)}):')
    for c in data_cols:
        if not df.empty:
            non_null = df[c].apply(lambda v: not is_missing(v)).sum()
            print(f'    {c:<60s} {non_null:>4d}/{len(df)} non-null')
        else:
            print(f'    {c}')
    if not data_cols:
        print('  ⚠️  没有 summary/* 数据列!')
    print()


def copy_run(src_run, dst_run_id=None):
    """
    将源 run 的 summary/* 表格数据复制到目标项目中同名 run。
    dst_run_id: 如果目标 run 已存在, 传入其 id 来 resume。
    """
    print(f'\n{"="*72}')
    print(f'  处理: {src_run.name} ({src_run.id})')

    df, data_cols = fetch_summary_table(src_run)

    if df.empty:
        print(f'  ⚠️  跳过: 没有 summary/* 数据')
        return

    # wandb.init 参数
    init_kwargs = {
        'entity': ENTITY,
        'project': DST_PROJECT,
        'name': src_run.name,
    }

    if dst_run_id:
        init_kwargs['id'] = dst_run_id
        init_kwargs['resume'] = 'must'
        print(f'  resuming: {dst_run_id}')
    else:
        init_kwargs['tags'] = list(src_run.tags) + [f'copied-from:{SRC_PROJECT}']
        init_kwargs['config'] = dict(src_run.config)
        print(f'  creating new run')

    new_run = wandb.init(**init_kwargs)
    print(f'  url: {new_run.url}')

    # 横轴 = epoch
    wandb.define_metric(STEP_METRIC)
    wandb.define_metric('*', step_metric=STEP_METRIC)

    # 逐行上传 (每行 = 一个 epoch)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
        log_dict = {
            col: val for col, val in row.items()
            if not is_missing(val)
        }
        if log_dict:
            wandb.log(log_dict)

    wandb.finish()
    print(f'  ✅ done: {src_run.name} ({len(df)} rows, {len(data_cols)} cols)')


print('工具函数已定义 ✓')

工具函数已定义 ✓


In [4]:
# ── Step 0: 列出源项目所有 run ─────────────────────────────────────
api = wandb.Api()
src_runs = list(api.runs(f'{ENTITY}/{SRC_PROJECT}'))
print(f'源项目 {SRC_PROJECT} 共有 {len(src_runs)} 个 runs:\n')
for i, r in enumerate(src_runs):
    print(f'  [{i:2d}] {r.name:<30s}  id={r.id}  state={r.state}')

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


源项目 eval_log 共有 9 个 runs:

  [ 0] model_1226                      id=5wljyc8f  state=finished
  [ 1] s3_0227                         id=amd2hdi8  state=finished
  [ 2] s3_12_13                        id=w00uwrr5  state=finished
  [ 3] s3_12_21                        id=wd5v1ilg  state=finished
  [ 4] s3_12_26                        id=c6j2p5p1  state=finished
  [ 5] s3_12_29                        id=z7u615bs  state=finished
  [ 6] s3_01_04                        id=ndvogudv  state=finished
  [ 7] s3_0315                         id=vj0j1ixn  state=finished
  [ 8] s3_0925                         id=8amsml4s  state=finished


In [5]:
# ── Step 1: 预览所有 run ─────────────────────────────────────────
# 先看看将要复制的内容, 确认无误后再执行下一个 cell。

api = wandb.Api()
src_runs = list(api.runs(f'{ENTITY}/{SRC_PROJECT}'))

# 缓存目标项目已有 run 名
dst_run_names = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

print(f'源项目 {SRC_PROJECT}: {len(src_runs)} runs')
print(f'目标项目 {DST_PROJECT}: {len(dst_run_names)} existing runs')
print(f'排除列: {EXCLUDE_COLUMNS}')
print()

for src_run in tqdm(src_runs, desc='预览进度'):
    preview_run(src_run, src_run.name in dst_run_names)

源项目 eval_log: 9 runs
目标项目 eval_all: 0 existing runs
排除列: {'summary/IJBC_TPR@FPR0.01'}



预览进度:   0%|          | 0/9 [00:00<?, ?it/s]

  run: model_1226 (5wljyc8f)
  dst: create new
  epoch 数: 21
  data columns (24):
    summary/ijbc_001_tpir_at_far_1e-05                             21/21 non-null
    summary/ijbc_001_tpir_at_far_1e-06                             21/21 non-null
    summary/ijbc_001_tpir_at_far_1e-07                             21/21 non-null
    summary/ijbc_001_tpir_at_far_1e-08                             21/21 non-null
    summary/ijbc_001_tpir_at_far_1e-09                             21/21 non-null
    summary/ijbc_001_tpir_at_far_1e-10                             21/21 non-null
    summary/ijbc_001_tpir_at_far_5e-07                             21/21 non-null
    summary/ijbc_all_tpir_at_far_1e-05                             21/21 non-null
    summary/ijbc_all_tpir_at_far_1e-06                             21/21 non-null
    summary/ijbc_all_tpir_at_far_1e-07                             21/21 non-null
    summary/ijbc_all_tpir_at_far_1e-08                             21/21 non-null
    summary/ijbc

In [6]:
# ── Step 2: 执行复制 ─────────────────────────────────────────────
# 确认 preview 没问题后, 执行这个 cell 真正写入。

api = wandb.Api()
src_runs = list(api.runs(f'{ENTITY}/{SRC_PROJECT}'))
dst_run_names = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

print(f'即将复制 {len(src_runs)} 个 runs')
print(f'  {SRC_PROJECT} → {DST_PROJECT}')
print(f'  排除列: {EXCLUDE_COLUMNS}')
print()

for i, src_run in enumerate(src_runs):
    print(f'\n[{i+1}/{len(src_runs)}]')
    copy_run(src_run, dst_run_id=dst_run_names.get(src_run.name))

print('\n' + '=' * 72)
print('🎉 全部完成!')

即将复制 9 个 runs
  eval_log → eval_all
  排除列: {'summary/IJBC_TPR@FPR0.01'}


[1/9]

  处理: model_1226 (5wljyc8f)
  creating new run


wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  url: https://wandb.ai/kejian-zhao-tsinghua-university/eval_all/runs/xg6vn046


  uploading:   0%|          | 0/21 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
summary/ijbc_001_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_all_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_all_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+15,...


...

  ✅ done: s3_0315 (24 rows, 24 cols)

[9/9]

  处理: s3_0925 (8amsml4s)
  creating new run


  url: https://wandb.ai/kejian-zhao-tsinghua-university/eval_all/runs/xxrcuxlm


  uploading:   0%|          | 0/21 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
summary/ijbc_001_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_all_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_all_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+15,...


  ✅ done: s3_0925 (21 rows, 24 cols)

🎉 全部完成!


In [7]:
# ── 单独复制 summary/IJBC_TPR@FPR0.01 到 eval_all ─────────────────
# 目标 run: s3_0315, s3_0925

TARGET_COL = 'summary/IJBC_TPR@FPR0.01'
RUN_NAMES = ['s3_0315', 's3_0925']

api = wandb.Api()
dst_run_names = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

for run_name in RUN_NAMES:
    src_run = [r for r in api.runs(f'{ENTITY}/{SRC_PROJECT}') if r.name == run_name]
    assert len(src_run) == 1, f'找不到或有重名: {run_name}'
    src_run = src_run[0]

    # 读取 history
    rows = list(src_run.scan_history())
    df = pd.DataFrame(rows)
    keep = [c for c in [STEP_METRIC, TARGET_COL] if c in df.columns]
    df = df[keep].dropna(subset=[TARGET_COL], how='all').reset_index(drop=True)

    print(f'\n{"="*72}')
    print(f'  run: {run_name} ({src_run.id})')
    print(f'  列: {TARGET_COL}')
    print(f'  epoch 数: {len(df)}')

    # 写入目标
    init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': run_name}
    if run_name in dst_run_names:
        init_kwargs['id'] = dst_run_names[run_name]
        init_kwargs['resume'] = 'must'
        print(f'  resuming: {dst_run_names[run_name]}')
    else:
        init_kwargs['tags'] = list(src_run.tags) + [f'copied-from:{SRC_PROJECT}']
        init_kwargs['config'] = dict(src_run.config)
        print(f'  creating new run')

    wandb.init(**init_kwargs)
    wandb.define_metric(STEP_METRIC)
    wandb.define_metric('*', step_metric=STEP_METRIC)

    for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
        log_dict = {col: val for col, val in row.items() if not is_missing(val)}
        if log_dict:
            wandb.log(log_dict)

    wandb.finish()
    print(f'  ✅ done: {run_name} ({len(df)} rows)')


  run: s3_0315 (vj0j1ixn)
  列: summary/IJBC_TPR@FPR0.01
  epoch 数: 24
  resuming: e33h0f3i


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  uploading:   0%|          | 0/24 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
summary/IJBC_TPR@FPR0.01,█▇▆▆▅▅▅▅▄▄▄▃▃▃▂▂▂▂▂▁▂▁▁▁
epoch,23
summary/IJBC_TPR@FPR0.01,92.45794
summary/ijbc_001_tpir_at_far_1e-05,85.58881
summary/ijbc_001_tpir_at_far_1e-06,62.9198
summary/ijbc_001_tpir_at_far_1e-07,7.49195
summary/ijbc_001_tpir_at_far_1e-08,0.59336
summary/ijbc_001_tpir_at_far_1e-09,0.59336
summary/ijbc_001_tpir_at_far_1e-10,0.59336
summary/ijbc_001_tpir_at_far_5e-07,36.24693


  ✅ done: s3_0315 (24 rows)

  run: s3_0925 (8amsml4s)
  列: summary/IJBC_TPR@FPR0.01
  epoch 数: 21
  resuming: xxrcuxlm


  uploading:   0%|          | 0/21 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
summary/IJBC_TPR@FPR0.01,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,20
summary/IJBC_TPR@FPR0.01,88.96559
summary/ijbc_001_tpir_at_far_1e-05,82.29181
summary/ijbc_001_tpir_at_far_1e-06,66.37701
summary/ijbc_001_tpir_at_far_1e-07,12.21436
summary/ijbc_001_tpir_at_far_1e-08,0.59346
summary/ijbc_001_tpir_at_far_1e-09,0.59346
summary/ijbc_001_tpir_at_far_1e-10,0.59346
summary/ijbc_001_tpir_at_far_5e-07,46.29302


  ✅ done: s3_0925 (21 rows)


In [10]:
# ── 从 work_1201 复制 summary/IJBC_TPR@FPR0.01 到 eval_all ────────

SRC_PROJECT_1201 = 'work_1201'
TARGET_COL = 'summary/IJBC_TPR@FPR0.01'

RUN_MAP = {
    'ft_ir101_s3_full_12-13_0': 's3_12_13',
    'ft_ir101_s3_full_12-21_0': 's3_12_21',
    'ft_ir101_s3_full_12-26_0': 's3_12_26',
    'ft_ir101_s3_full_12-29_0': 's3_12_29',
}

api = wandb.Api()
src_all_runs = {r.name: r for r in api.runs(f'{ENTITY}/{SRC_PROJECT_1201}')}
dst_run_names = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

for src_name, dst_name in RUN_MAP.items():
    assert src_name in src_all_runs, f'找不到: {src_name}'
    src_run = src_all_runs[src_name]

    rows = list(src_run.scan_history(keys=[STEP_METRIC, TARGET_COL]))
    df = pd.DataFrame(rows)
    df = df.dropna(subset=[TARGET_COL], how='all').reset_index(drop=True)

    print(f'\n{"="*72}')
    print(f'  {src_name} → {dst_name}')
    print(f'  epoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')

    init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': dst_name}
    if dst_name in dst_run_names:
        init_kwargs['id'] = dst_run_names[dst_name]
        init_kwargs['resume'] = 'must'
        print(f'  resuming: {dst_run_names[dst_name]}')
    else:
        init_kwargs['tags'] = [f'copied-from:{SRC_PROJECT_1201}']
        init_kwargs['config'] = dict(src_run.config)
        print(f'  creating new run')

    wandb.init(**init_kwargs)
    wandb.define_metric(STEP_METRIC)
    wandb.define_metric('*', step_metric=STEP_METRIC)

    for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
        log_dict = {col: val for col, val in row.items() if not is_missing(val)}
        if log_dict:
            wandb.log(log_dict)

    wandb.finish()
    print(f'  ✅ done: {dst_name} ({len(df)} rows)')


  ft_ir101_s3_full_12-13_0 → s3_12_13
  epoch 数: 26  |  epochs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]
  resuming: wk2dqdam


  uploading:   0%|          | 0/26 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
summary/IJBC_TPR@FPR0.01,██▇▆▆▅▄▃▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,25
summary/IJBC_TPR@FPR0.01,93.55218
summary/ijbc_001_tpir_at_far_1e-05,90.70655
summary/ijbc_001_tpir_at_far_1e-06,72.77777
summary/ijbc_001_tpir_at_far_1e-07,11.48847
summary/ijbc_001_tpir_at_far_1e-08,0.59401
summary/ijbc_001_tpir_at_far_1e-09,0.59401
summary/ijbc_001_tpir_at_far_1e-10,0.59401
summary/ijbc_001_tpir_at_far_5e-07,48.31278


  ✅ done: s3_12_13 (26 rows)

  ft_ir101_s3_full_12-21_0 → s3_12_21
  epoch 数: 18  |  epochs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]
  resuming: 99esskr4


  uploading:   0%|          | 0/18 [00:00<?, ?it/s]

...

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
summary/IJBC_TPR@FPR0.01,█▇▇▇▇▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁
epoch,29
summary/IJBC_TPR@FPR0.01,94.03794
summary/ijbc_001_tpir_at_far_1e-05,91.63337
summary/ijbc_001_tpir_at_far_1e-06,73.32646
summary/ijbc_001_tpir_at_far_1e-07,11.45763
summary/ijbc_001_tpir_at_far_1e-08,0.59401
summary/ijbc_001_tpir_at_far_1e-09,0.59401
summary/ijbc_001_tpir_at_far_1e-10,0.59401
summary/ijbc_001_tpir_at_far_5e-07,48.52434


  ✅ done: s3_12_26 (30 rows)

  ft_ir101_s3_full_12-29_0 → s3_12_29
  epoch 数: 19  |  epochs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
  resuming: 0w0bc6ho


  uploading:   0%|          | 0/19 [00:00<?, ?it/s]

epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
summary/IJBC_TPR@FPR0.01,█▇█▇█▅▆▅▅▅▄▃▂▅▃▃▁▂▂
epoch,18
summary/IJBC_TPR@FPR0.01,93.91011
summary/ijbc_001_tpir_at_far_1e-05,91.50803
summary/ijbc_001_tpir_at_far_1e-06,73.3504
summary/ijbc_001_tpir_at_far_1e-07,11.2224
summary/ijbc_001_tpir_at_far_1e-08,0.59401
summary/ijbc_001_tpir_at_far_1e-09,0.59401
summary/ijbc_001_tpir_at_far_1e-10,0.59401
summary/ijbc_001_tpir_at_far_5e-07,48.77671


  ✅ done: s3_12_29 (19 rows)


In [11]:
# ── 从 work_0101 复制 summary/IJBC_TPR@FPR0.01 到 eval_all ────────

SRC_PRJ = 'work_0101'
TARGET_COL = 'summary/IJBC_TPR@FPR0.01'
RUN_MAP_0101 = {
    'ft_ir101_s3_full_01-04_0': 's3_01_04',
}

api = wandb.Api()
src_all = {r.name: r for r in api.runs(f'{ENTITY}/{SRC_PRJ}')}
dst_all = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

for src_name, dst_name in RUN_MAP_0101.items():
    assert src_name in src_all, f'找不到: {src_name}'
    src_run = src_all[src_name]

    rows = list(src_run.scan_history(keys=[STEP_METRIC, TARGET_COL]))
    df = pd.DataFrame(rows)
    df = df.dropna(subset=[TARGET_COL], how='all').reset_index(drop=True)

    print(f'\n{"="*72}')
    print(f'  {src_name} → {dst_name}')
    print(f'  epoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')

    init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': dst_name}
    if dst_name in dst_all:
        init_kwargs['id'] = dst_all[dst_name]
        init_kwargs['resume'] = 'must'
        print(f'  resuming: {dst_all[dst_name]}')
    else:
        init_kwargs['tags'] = [f'copied-from:{SRC_PRJ}']
        init_kwargs['config'] = dict(src_run.config)
        print(f'  creating new run')

    wandb.init(**init_kwargs)
    wandb.define_metric(STEP_METRIC)
    wandb.define_metric('*', step_metric=STEP_METRIC)

    for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
        log_dict = {col: val for col, val in row.items() if not is_missing(val)}
        if log_dict:
            wandb.log(log_dict)

    wandb.finish()
    print(f'  ✅ done: {dst_name} ({len(df)} rows)')


  ft_ir101_s3_full_01-04_0 → s3_01_04
  epoch 数: 16  |  epochs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  resuming: 9q3hl6hm


  uploading:   0%|          | 0/16 [00:00<?, ?it/s]

epoch,▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
summary/IJBC_TPR@FPR0.01,█▇▆▅▅▄▄▃▃▃▁▁▂▁▂▂
epoch,15
summary/IJBC_TPR@FPR0.01,91.96707
summary/ijbc_001_tpir_at_far_1e-05,88.46861
summary/ijbc_001_tpir_at_far_1e-06,69.16275
summary/ijbc_001_tpir_at_far_1e-07,11.29482
summary/ijbc_001_tpir_at_far_1e-08,0.59401
summary/ijbc_001_tpir_at_far_1e-09,0.59401
summary/ijbc_001_tpir_at_far_1e-10,0.59401
summary/ijbc_001_tpir_at_far_5e-07,43.80823


  ✅ done: s3_01_04 (16 rows)


In [4]:
# ── 从 work_0213_eval_all_s2 复制 s3_0320 → eval_all (s3_03_20) ──
# summary/work_tpir_at_far_1e* → summary/work_0213_tpir_at_far_1e*
# 按 epoch 复制所有 summary/* 数据

SRC_PRJ_0213 = 'work_0213_eval_all_s2'
SRC_RUN_NAME = 's3_0320'
DST_RUN_NAME = 's3_03_20'
RENAME_PREFIX_OLD = 'summary/work_tpir_at_far_1e'
RENAME_PREFIX_NEW = 'summary/work_0213_tpir_at_far_1e'

api = wandb.Api()

# 找到源 run
src_runs_list = [r for r in api.runs(f'{ENTITY}/{SRC_PRJ_0213}') if r.name == SRC_RUN_NAME]
assert len(src_runs_list) == 1, f'找不到或有重名: {SRC_RUN_NAME} in {SRC_PRJ_0213}'
src_run = src_runs_list[0]

# 获取所有 summary/* 列
all_cols = src_run.history(samples=1).columns.tolist()
data_cols = sorted([c for c in all_cols if c.startswith(COPY_PREFIX)])
print(f'源 run: {src_run.name} ({src_run.id})')
print(f'summary/* 列 ({len(data_cols)}):')
for c in data_cols:
    renamed = c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW) if c.startswith(RENAME_PREFIX_OLD) else c
    tag = f'  → {renamed}' if renamed != c else ''
    print(f'  {c}{tag}')

# 拉取数据
rows = list(src_run.scan_history(keys=[STEP_METRIC] + data_cols))
df = pd.DataFrame(rows)
keep = [c for c in [STEP_METRIC] + data_cols if c in df.columns]
df = df[keep].dropna(subset=data_cols, how='all').reset_index(drop=True)

# 重命名列: summary/work_tpir_at_far_1e* → summary/work_0213_tpir_at_far_1e*
rename_map = {
    c: c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW)
    for c in df.columns if c.startswith(RENAME_PREFIX_OLD)
}
if rename_map:
    df = df.rename(columns=rename_map)
    print(f'\n重命名 {len(rename_map)} 列:')
    for old, new in rename_map.items():
        print(f'  {old} → {new}')

print(f'\nepoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')

# 目标项目已有 run
dst_all = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

print(f'\n{"="*72}')
print(f'  {SRC_RUN_NAME} ({SRC_PRJ_0213}) → {DST_RUN_NAME} ({DST_PROJECT})')

init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': DST_RUN_NAME}
if DST_RUN_NAME in dst_all:
    init_kwargs['id'] = dst_all[DST_RUN_NAME]
    init_kwargs['resume'] = 'must'
    print(f'  resuming: {dst_all[DST_RUN_NAME]}')
else:
    init_kwargs['tags'] = list(src_run.tags) + [f'copied-from:{SRC_PRJ_0213}']
    init_kwargs['config'] = dict(src_run.config)
    print(f'  creating new run')

wandb.init(**init_kwargs)
wandb.define_metric(STEP_METRIC)
wandb.define_metric('*', step_metric=STEP_METRIC)

for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
    log_dict = {col: val for col, val in row.items() if not is_missing(val)}
    if log_dict:
        wandb.log(log_dict)

wandb.finish()
print(f'  done: {DST_RUN_NAME} ({len(df)} rows, {len(df.columns)-1} data cols)')

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


源 run: s3_0320 (3apfn7yb)
summary/* 列 (25):
  summary/IJBC_TPR@FPR0.01
  summary/ijbc_001_tpir_at_far_1e-05
  summary/ijbc_001_tpir_at_far_1e-06
  summary/ijbc_001_tpir_at_far_1e-07
  summary/ijbc_001_tpir_at_far_1e-08
  summary/ijbc_001_tpir_at_far_1e-09
  summary/ijbc_001_tpir_at_far_1e-10
  summary/ijbc_001_tpir_at_far_5e-07
  summary/ijbc_all_tpir_at_far_1e-05
  summary/ijbc_all_tpir_at_far_1e-06
  summary/ijbc_all_tpir_at_far_1e-07
  summary/ijbc_all_tpir_at_far_1e-08
  summary/ijbc_all_tpir_at_far_1e-09
  summary/ijbc_all_tpir_at_far_1e-10
  summary/ijbc_all_tpir_at_far_5e-07
  summary/work_1201_tpir_at_far_1e-06
  summary/work_1201_tpir_at_far_1e-07
  summary/work_1201_tpir_at_far_1e-08
  summary/work_1201_tpir_at_far_1e-09
  summary/work_1201_tpir_at_far_1e-10
  summary/work_tpir_at_far_1e-06  → summary/work_0213_tpir_at_far_1e-06
  summary/work_tpir_at_far_1e-07  → summary/work_0213_tpir_at_far_1e-07
  summary/work_tpir_at_far_1e-08  → summary/work_0213_tpir_at_far_1e-08
  sum

wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  uploading:   0%|          | 0/10 [00:00<?, ?it/s]

epoch,▁▂▃▃▄▅▆▆▇█
summary/IJBC_TPR@FPR0.01,█▆▃▅▁▂▃▂▃▂
summary/ijbc_001_tpir_at_far_1e-05,█▅▄▃▁▁▃▂▃▂
summary/ijbc_001_tpir_at_far_1e-06,██▅▇▁▇▆▆▆▆
summary/ijbc_001_tpir_at_far_1e-07,▆█▆▅▁▅▅▅▄▅
summary/ijbc_001_tpir_at_far_1e-08,█▃▂▆▁▂▂▄▆▂
summary/ijbc_001_tpir_at_far_1e-09,█▃▂▆▁▂▂▄▆▂
summary/ijbc_001_tpir_at_far_1e-10,█▃▂▆▁▂▂▄▆▂
summary/ijbc_001_tpir_at_far_5e-07,█▇▅█▁▆▅▅▅▅
summary/ijbc_all_tpir_at_far_1e-05,▇█▅▃▁▃▃▃▃▃
+16,...


  done: s3_03_20 (10 rows, 25 data cols)


In [5]:
# ── 从 work_0320_eval 复制 s3_0322 → eval_all (s3_03_22) ──
# 按 epoch 复制所有 summary/* 数据

SRC_PRJ_0213 = 'work_0320_eval'
SRC_RUN_NAME = 's2_0322'
DST_RUN_NAME = 's2_03_22'
RENAME_PREFIX_OLD = 'summary/work_tpir_at_far_1e'
RENAME_PREFIX_NEW = 'summary/work_0213_tpir_at_far_1e'

api = wandb.Api()

# 找到源 run
src_runs_list = [r for r in api.runs(f'{ENTITY}/{SRC_PRJ_0213}') if r.name == SRC_RUN_NAME]
assert len(src_runs_list) == 1, f'找不到或有重名: {SRC_RUN_NAME} in {SRC_PRJ_0213}'
src_run = src_runs_list[0]

# 获取所有 summary/* 列
all_cols = src_run.history(samples=1).columns.tolist()
data_cols = sorted([c for c in all_cols if c.startswith(COPY_PREFIX)])
print(f'源 run: {src_run.name} ({src_run.id})')
print(f'summary/* 列 ({len(data_cols)}):')
for c in data_cols:
    renamed = c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW) if c.startswith(RENAME_PREFIX_OLD) else c
    tag = f'  → {renamed}' if renamed != c else ''
    print(f'  {c}{tag}')

# 拉取数据
rows = list(src_run.scan_history(keys=[STEP_METRIC] + data_cols))
df = pd.DataFrame(rows)
keep = [c for c in [STEP_METRIC] + data_cols if c in df.columns]
df = df[keep].dropna(subset=data_cols, how='all').reset_index(drop=True)

# 重命名列: summary/work_tpir_at_far_1e* → summary/work_0213_tpir_at_far_1e*
rename_map = {
    c: c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW)
    for c in df.columns if c.startswith(RENAME_PREFIX_OLD)
}
if rename_map:
    df = df.rename(columns=rename_map)
    print(f'\n重命名 {len(rename_map)} 列:')
    for old, new in rename_map.items():
        print(f'  {old} → {new}')

print(f'\nepoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')

# 目标项目已有 run
dst_all = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

print(f'\n{"="*72}')
print(f'  {SRC_RUN_NAME} ({SRC_PRJ_0213}) → {DST_RUN_NAME} ({DST_PROJECT})')

init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': DST_RUN_NAME}
if DST_RUN_NAME in dst_all:
    init_kwargs['id'] = dst_all[DST_RUN_NAME]
    init_kwargs['resume'] = 'must'
    print(f'  resuming: {dst_all[DST_RUN_NAME]}')
else:
    init_kwargs['tags'] = list(src_run.tags) + [f'copied-from:{SRC_PRJ_0213}']
    init_kwargs['config'] = dict(src_run.config)
    print(f'  creating new run')

wandb.init(**init_kwargs)
wandb.define_metric(STEP_METRIC)
wandb.define_metric('*', step_metric=STEP_METRIC)

for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
    log_dict = {col: val for col, val in row.items() if not is_missing(val)}
    if log_dict:
        wandb.log(log_dict)

wandb.finish()
print(f'  done: {DST_RUN_NAME} ({len(df)} rows, {len(df.columns)-1} data cols)')

源 run: s2_0322 (eko41onc)
summary/* 列 (35):
  summary/IJBC_TPR@FPR0.01
  summary/ijbc_001_tpir_at_far_1e-05
  summary/ijbc_001_tpir_at_far_1e-06
  summary/ijbc_001_tpir_at_far_1e-07
  summary/ijbc_001_tpir_at_far_1e-08
  summary/ijbc_001_tpir_at_far_1e-09
  summary/ijbc_001_tpir_at_far_1e-10
  summary/ijbc_001_tpir_at_far_5e-07
  summary/ijbc_all_tpir_at_far_1e-05
  summary/ijbc_all_tpir_at_far_1e-06
  summary/ijbc_all_tpir_at_far_1e-07
  summary/ijbc_all_tpir_at_far_1e-08
  summary/ijbc_all_tpir_at_far_1e-09
  summary/ijbc_all_tpir_at_far_1e-10
  summary/ijbc_all_tpir_at_far_5e-07
  summary/work_0320_3t_tpir_at_far_1e-06
  summary/work_0320_3t_tpir_at_far_1e-07
  summary/work_0320_3t_tpir_at_far_1e-08
  summary/work_0320_3t_tpir_at_far_1e-09
  summary/work_0320_3t_tpir_at_far_1e-10
  summary/work_0320_glint_tpir_at_far_1e-06
  summary/work_0320_glint_tpir_at_far_1e-07
  summary/work_0320_glint_tpir_at_far_1e-08
  summary/work_0320_glint_tpir_at_far_1e-09
  summary/work_0320_glint_tpir

wandb: Currently logged in as: kejian-zhao (kejian-zhao-tsinghua-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  uploading:   0%|          | 0/20 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
summary/IJBC_TPR@FPR0.01,▆▁▄▆▅▅▃▄▄▅▆▅█▇▇▇▇█▇█
summary/ijbc_001_tpir_at_far_1e-05,▇▁▅▃▇▂▆▃▃▃█▄▄▆▅█▆█▇▇
summary/ijbc_001_tpir_at_far_1e-06,█▁▅▆▇▄▆▅▆▇▄▄▅▃▅▄▅▄▄▄
summary/ijbc_001_tpir_at_far_1e-07,▅▁▅▆▆▄▇▆▅█▄▅▅▅▆▅▆▆▆▆
summary/ijbc_001_tpir_at_far_1e-08,▆▁▅▇██▇▇▃▂▄▅▅▅▃▃▅▆▆▆
summary/ijbc_001_tpir_at_far_1e-09,▆▁▅▇██▇▇▃▂▄▅▅▅▃▃▅▆▆▆
summary/ijbc_001_tpir_at_far_1e-10,▆▁▅▇██▇▇▃▂▄▅▅▅▃▃▅▆▆▆
summary/ijbc_001_tpir_at_far_5e-07,█▁▆██▅▆▆▆▇▄▄▅▄▆▅▆▆▅▆
summary/ijbc_all_tpir_at_far_1e-05,█▁▂▄▆▃▃▃▄▅▂▂▂▂▃▂▃▃▂▃
+26,...


  done: s2_03_22 (20 rows, 35 data cols)


In [5]:
# ── 将 work_0320_eval/s3_0925 的 epoch 0 复制 20 份 (epoch 0~19) ──
# 目的: 只有 1 个 epoch 结果, 复制成 20 份画一条水平直线

import math
import pandas as pd
import wandb
from tqdm.auto import tqdm

ENTITY = 'kejian-zhao-tsinghua-university'
TARGET_PROJECT = 'work_0320_eval'
RUN_NAME = 's3_0925'
COPY_PREFIX = 'summary/'
STEP_METRIC = 'epoch'
NUM_EPOCHS = 20  # 复制成 20 份 (epoch 0 ~ 19)

def is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    try:
        m = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(m) if isinstance(m, bool) else False

api = wandb.Api()

# 找到目标 run
runs_list = [r for r in api.runs(f'{ENTITY}/{TARGET_PROJECT}') if r.name == RUN_NAME]
assert len(runs_list) == 1, f'找不到或有重名: {RUN_NAME}'
src_run = runs_list[0]
run_id = src_run.id

print(f'目标 run: {RUN_NAME} (id={run_id})')
print(f'项目: {TARGET_PROJECT}')

# 获取 summary/* 列
all_cols = src_run.history(samples=1).columns.tolist()
data_cols = sorted([c for c in all_cols if c.startswith(COPY_PREFIX)])
print(f'\nsummary/* 列 ({len(data_cols)}):')
for c in data_cols:
    print(f'  {c}')

# 读取 epoch 0 的数据
rows = list(src_run.scan_history(keys=[STEP_METRIC] + data_cols))
df = pd.DataFrame(rows)
keep = [c for c in [STEP_METRIC] + data_cols if c in df.columns]
df = df[keep].dropna(subset=data_cols, how='all').reset_index(drop=True)

print(f'\n原始 epoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')
assert len(df) == 1, f'预期只有 1 个 epoch, 实际有 {len(df)} 个'

# 取 epoch 0 的数据行 (不含 epoch 列)
epoch0_data = {col: df.iloc[0][col] for col in data_cols if not is_missing(df.iloc[0][col])}
print(f'\nepoch 0 非空数据列: {len(epoch0_data)}')
for col, val in sorted(epoch0_data.items()):
    print(f'  {col}: {val}')

# 构造 20 行: epoch 0~19, 每行数据完全相同
print(f'\n{"="*72}')
print(f'  将 epoch 0 的数据复制 {NUM_EPOCHS} 份 (epoch 0~{NUM_EPOCHS-1})')
print(f'  resume run: {run_id}')

wandb.init(
    entity=ENTITY,
    project=TARGET_PROJECT,
    id=run_id,
    resume='must',
)
wandb.define_metric(STEP_METRIC)
wandb.define_metric('*', step_metric=STEP_METRIC)

for ep in tqdm(range(NUM_EPOCHS), desc='  uploading'):
    log_dict = {STEP_METRIC: ep, **epoch0_data}
    wandb.log(log_dict)

wandb.finish()
print(f'  ✅ done: {RUN_NAME} — {NUM_EPOCHS} epochs 写入完成 (水平直线)')

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


目标 run: s3_0925 (id=4js07zno)
项目: work_0320_eval

summary/* 列 (35):
  summary/IJBC_TPR@FPR0.01
  summary/ijbc_001_tpir_at_far_1e-05
  summary/ijbc_001_tpir_at_far_1e-06
  summary/ijbc_001_tpir_at_far_1e-07
  summary/ijbc_001_tpir_at_far_1e-08
  summary/ijbc_001_tpir_at_far_1e-09
  summary/ijbc_001_tpir_at_far_1e-10
  summary/ijbc_001_tpir_at_far_5e-07
  summary/ijbc_all_tpir_at_far_1e-05
  summary/ijbc_all_tpir_at_far_1e-06
  summary/ijbc_all_tpir_at_far_1e-07
  summary/ijbc_all_tpir_at_far_1e-08
  summary/ijbc_all_tpir_at_far_1e-09
  summary/ijbc_all_tpir_at_far_1e-10
  summary/ijbc_all_tpir_at_far_5e-07
  summary/work_0320_3t_tpir_at_far_1e-06
  summary/work_0320_3t_tpir_at_far_1e-07
  summary/work_0320_3t_tpir_at_far_1e-08
  summary/work_0320_3t_tpir_at_far_1e-09
  summary/work_0320_3t_tpir_at_far_1e-10
  summary/work_0320_glint_tpir_at_far_1e-06
  summary/work_0320_glint_tpir_at_far_1e-07
  summary/work_0320_glint_tpir_at_far_1e-08
  summary/work_0320_glint_tpir_at_far_1e-09
  summ

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


  uploading:   0%|          | 0/20 [00:00<?, ?it/s]

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
summary/IJBC_TPR@FPR0.01,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-06,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-08,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-09,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_1e-10,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_001_tpir_at_far_5e-07,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
summary/ijbc_all_tpir_at_far_1e-05,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+26,...


  ✅ done: s3_0925 — 20 epochs 写入完成 (水平直线)


In [7]:
# ── 配置 ──────────────────────────────────────────────────────────
ENTITY = 'kejian-zhao-tsinghua-university'
SRC_PROJECT = 'work_0320_eval'
DST_PROJECT = 'eval_all'

# 只复制以此开头的 history 列 (即 W&B 面板上的表格/折线图)
COPY_PREFIX = 'summary/'

# 要排除的列名 (精确匹配)
EXCLUDE_COLUMNS = {
    # 'summary/IJBC_TPR@FPR0.01',
}

# 横轴: epoch
STEP_METRIC = 'epoch'

print(f'源项目  : {ENTITY}/{SRC_PROJECT}')
print(f'目标项目: {ENTITY}/{DST_PROJECT}')
print(f'复制前缀: {COPY_PREFIX}*')
print(f'排除列  : {EXCLUDE_COLUMNS}')
print(f'横轴    : {STEP_METRIC}')

# ── 从 work_0320_eval 复制 s3_0322 → eval_all (s3_03_22) ──
# 按 epoch 复制所有 summary/* 数据

SRC_PRJ_0213 = 'work_0320_eval'
SRC_RUN_NAME = 's3_0323'
DST_RUN_NAME = 's3_03_23'
RENAME_PREFIX_OLD = 'null'
RENAME_PREFIX_NEW = 'null'

api = wandb.Api()

# 找到源 run
src_runs_list = [r for r in api.runs(f'{ENTITY}/{SRC_PRJ_0213}') if r.name == SRC_RUN_NAME]
assert len(src_runs_list) == 1, f'找不到或有重名: {SRC_RUN_NAME} in {SRC_PRJ_0213}'
src_run = src_runs_list[0]

# 获取所有 summary/* 列
all_cols = src_run.history(samples=1).columns.tolist()
data_cols = sorted([c for c in all_cols if c.startswith(COPY_PREFIX)])
print(f'源 run: {src_run.name} ({src_run.id})')
print(f'summary/* 列 ({len(data_cols)}):')
for c in data_cols:
    renamed = c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW) if c.startswith(RENAME_PREFIX_OLD) else c
    tag = f'  → {renamed}' if renamed != c else ''
    print(f'  {c}{tag}')

# 拉取数据
rows = list(src_run.scan_history(keys=[STEP_METRIC] + data_cols))
df = pd.DataFrame(rows)
keep = [c for c in [STEP_METRIC] + data_cols if c in df.columns]
df = df[keep].dropna(subset=data_cols, how='all').reset_index(drop=True)

# 重命名列: summary/work_tpir_at_far_1e* → summary/work_0213_tpir_at_far_1e*
rename_map = {
    c: c.replace(RENAME_PREFIX_OLD, RENAME_PREFIX_NEW)
    for c in df.columns if c.startswith(RENAME_PREFIX_OLD)
}
if rename_map:
    df = df.rename(columns=rename_map)
    print(f'\n重命名 {len(rename_map)} 列:')
    for old, new in rename_map.items():
        print(f'  {old} → {new}')

print(f'\nepoch 数: {len(df)}  |  epochs: {df[STEP_METRIC].tolist()}')

# 目标项目已有 run
dst_all = {r.name: r.id for r in api.runs(f'{ENTITY}/{DST_PROJECT}')}

print(f'\n{"="*72}')
print(f'  {SRC_RUN_NAME} ({SRC_PRJ_0213}) → {DST_RUN_NAME} ({DST_PROJECT})')

init_kwargs = {'entity': ENTITY, 'project': DST_PROJECT, 'name': DST_RUN_NAME}
if DST_RUN_NAME in dst_all:
    init_kwargs['id'] = dst_all[DST_RUN_NAME]
    init_kwargs['resume'] = 'must'
    print(f'  resuming: {dst_all[DST_RUN_NAME]}')
else:
    init_kwargs['tags'] = list(src_run.tags) + [f'copied-from:{SRC_PRJ_0213}']
    init_kwargs['config'] = dict(src_run.config)
    print(f'  creating new run')

wandb.init(**init_kwargs)
wandb.define_metric(STEP_METRIC)
wandb.define_metric('*', step_metric=STEP_METRIC)

for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
    log_dict = {col: val for col, val in row.items() if not is_missing(val)}
    if log_dict:
        wandb.log(log_dict)

wandb.finish()
print(f'  done: {DST_RUN_NAME} ({len(df)} rows, {len(df.columns)-1} data cols)')

源项目  : kejian-zhao-tsinghua-university/work_0320_eval
目标项目: kejian-zhao-tsinghua-university/eval_all
复制前缀: summary/*
排除列  : {}
横轴    : epoch
源 run: s3_0323 (3lngf0p8)
summary/* 列 (35):
  summary/IJBC_TPR@FPR0.01
  summary/ijbc_001_tpir_at_far_1e-05
  summary/ijbc_001_tpir_at_far_1e-06
  summary/ijbc_001_tpir_at_far_1e-07
  summary/ijbc_001_tpir_at_far_1e-08
  summary/ijbc_001_tpir_at_far_1e-09
  summary/ijbc_001_tpir_at_far_1e-10
  summary/ijbc_001_tpir_at_far_5e-07
  summary/ijbc_all_tpir_at_far_1e-05
  summary/ijbc_all_tpir_at_far_1e-06
  summary/ijbc_all_tpir_at_far_1e-07
  summary/ijbc_all_tpir_at_far_1e-08
  summary/ijbc_all_tpir_at_far_1e-09
  summary/ijbc_all_tpir_at_far_1e-10
  summary/ijbc_all_tpir_at_far_5e-07
  summary/work_0320_3t_tpir_at_far_1e-06
  summary/work_0320_3t_tpir_at_far_1e-07
  summary/work_0320_3t_tpir_at_far_1e-08
  summary/work_0320_3t_tpir_at_far_1e-09
  summary/work_0320_3t_tpir_at_far_1e-10
  summary/work_0320_glint_tpir_at_far_1e-06
  summary/work_0320_g

  uploading:   0%|          | 0/10 [00:00<?, ?it/s]

epoch,▁▂▃▃▄▅▆▆▇█
summary/IJBC_TPR@FPR0.01,▄▂▁▁▃▃▇▅█▇
summary/ijbc_001_tpir_at_far_1e-05,▄▂▃▁▆▇▆▇▇█
summary/ijbc_001_tpir_at_far_1e-06,▆▆▂▁█▃▅▄▆▅
summary/ijbc_001_tpir_at_far_1e-07,█▂▃▁▆▃▁▁▃▁
summary/ijbc_001_tpir_at_far_1e-08,▆▇▆▄▇▅▆▆█▁
summary/ijbc_001_tpir_at_far_1e-09,▆▇▆▄▇▅▆▆█▁
summary/ijbc_001_tpir_at_far_1e-10,▆▇▆▄▇▅▆▆█▁
summary/ijbc_001_tpir_at_far_5e-07,▆▅▁▃█▅▇▅▇▆
summary/ijbc_all_tpir_at_far_1e-05,█▄▃▁█▄▄▃▆▅
+26,...


  done: s3_03_23 (10 rows, 35 data cols)
